# Stripe Transaction Data Analysis

This notebook provides a comprehensive analysis of Stripe transaction data.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime, timedelta

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load Data

Load your Stripe transaction data from the `data/raw/` folder.

In [ ]:
# Load transaction data
# Adjust the filename to match your uploaded file
df = pd.read_csv('../data/raw/transactions.csv')

print(f"Loaded {len(df):,} transactions")
df.head()

## 2. Data Exploration

In [ ]:
# Basic info
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Statistical summary
df.describe()

## 3. Data Cleaning & Transformation

In [ ]:
# Common Stripe data transformations

# Convert timestamps (adjust column name as needed)
if 'created' in df.columns:
    df['created'] = pd.to_datetime(df['created'], unit='s')  # If Unix timestamp
    # Or use: df['created'] = pd.to_datetime(df['created'])  # If ISO format

# Convert amount from cents to dollars (if applicable)
if 'amount' in df.columns:
    df['amount_dollars'] = df['amount'] / 100

if 'fee' in df.columns:
    df['fee_dollars'] = df['fee'] / 100

if 'net' in df.columns:
    df['net_dollars'] = df['net'] / 100

# Extract date components (if created column exists)
if 'created' in df.columns:
    df['date'] = df['created'].dt.date
    df['year'] = df['created'].dt.year
    df['month'] = df['created'].dt.month
    df['day_of_week'] = df['created'].dt.day_name()
    df['hour'] = df['created'].dt.hour

print("Data cleaned and transformed")
df.head()

## 4. Revenue Analysis

In [ ]:
# Total revenue metrics
if 'amount_dollars' in df.columns:
    total_revenue = df['amount_dollars'].sum()
    avg_transaction = df['amount_dollars'].mean()
    median_transaction = df['amount_dollars'].median()
    
    print(f"Total Revenue: ${total_revenue:,.2f}")
    print(f"Average Transaction: ${avg_transaction:,.2f}")
    print(f"Median Transaction: ${median_transaction:,.2f}")
    print(f"Total Transactions: {len(df):,}")

In [ ]:
# Revenue by status (if status column exists)
if 'status' in df.columns and 'amount_dollars' in df.columns:
    revenue_by_status = df.groupby('status')['amount_dollars'].agg(['sum', 'count', 'mean'])
    revenue_by_status.columns = ['Total Revenue', 'Count', 'Average Amount']
    print(revenue_by_status)

## 5. Time Series Analysis

In [ ]:
# Daily revenue trend
if 'date' in df.columns and 'amount_dollars' in df.columns:
    daily_revenue = df.groupby('date')['amount_dollars'].sum().reset_index()
    
    plt.figure(figsize=(15, 6))
    plt.plot(daily_revenue['date'], daily_revenue['amount_dollars'], marker='o', linewidth=2)
    plt.title('Daily Revenue Trend', fontsize=16, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Revenue ($)', fontsize=12)
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Monthly revenue (if year and month exist)
if 'year' in df.columns and 'month' in df.columns and 'amount_dollars' in df.columns:
    monthly_revenue = df.groupby(['year', 'month'])['amount_dollars'].sum().reset_index()
    monthly_revenue['year_month'] = monthly_revenue['year'].astype(str) + '-' + monthly_revenue['month'].astype(str).str.zfill(2)
    
    plt.figure(figsize=(15, 6))
    plt.bar(monthly_revenue['year_month'], monthly_revenue['amount_dollars'])
    plt.title('Monthly Revenue', fontsize=16, fontweight='bold')
    plt.xlabel('Month', fontsize=12)
    plt.ylabel('Revenue ($)', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Transaction Distribution

In [ ]:
# Transaction amount distribution
if 'amount_dollars' in df.columns:
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.hist(df['amount_dollars'], bins=50, edgecolor='black')
    plt.title('Transaction Amount Distribution', fontsize=14, fontweight='bold')
    plt.xlabel('Amount ($)', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    
    plt.subplot(1, 2, 2)
    plt.boxplot(df['amount_dollars'])
    plt.title('Transaction Amount Box Plot', fontsize=14, fontweight='bold')
    plt.ylabel('Amount ($)', fontsize=12)
    
    plt.tight_layout()
    plt.show()

## 7. Customer Analysis

In [ ]:
# Customer metrics (if customer column exists)
if 'customer' in df.columns and 'amount_dollars' in df.columns:
    customer_stats = df.groupby('customer').agg({
        'amount_dollars': ['sum', 'count', 'mean']
    }).reset_index()
    customer_stats.columns = ['customer', 'total_spent', 'transaction_count', 'avg_transaction']
    customer_stats = customer_stats.sort_values('total_spent', ascending=False)
    
    print("Top 10 Customers by Revenue:")
    print(customer_stats.head(10))
    
    # Plot top 10 customers
    plt.figure(figsize=(12, 6))
    top_10 = customer_stats.head(10)
    plt.barh(range(len(top_10)), top_10['total_spent'])
    plt.yticks(range(len(top_10)), top_10['customer'])
    plt.xlabel('Total Revenue ($)', fontsize=12)
    plt.title('Top 10 Customers by Revenue', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 8. Currency Analysis

In [ ]:
# Revenue by currency (if currency column exists)
if 'currency' in df.columns and 'amount_dollars' in df.columns:
    currency_revenue = df.groupby('currency')['amount_dollars'].agg(['sum', 'count']).reset_index()
    currency_revenue.columns = ['Currency', 'Total Revenue', 'Count']
    print(currency_revenue)
    
    # Pie chart
    plt.figure(figsize=(10, 6))
    plt.pie(currency_revenue['Total Revenue'], labels=currency_revenue['Currency'], autopct='%1.1f%%')
    plt.title('Revenue by Currency', fontsize=14, fontweight='bold')
    plt.show()

## 9. Fee Analysis

In [ ]:
# Stripe fees analysis
if 'fee_dollars' in df.columns and 'amount_dollars' in df.columns:
    total_fees = df['fee_dollars'].sum()
    total_revenue = df['amount_dollars'].sum()
    avg_fee_rate = (total_fees / total_revenue * 100) if total_revenue > 0 else 0
    
    print(f"Total Fees Paid: ${total_fees:,.2f}")
    print(f"Average Fee Rate: {avg_fee_rate:.2f}%")
    
    # Calculate fee percentage for each transaction
    df['fee_percentage'] = (df['fee_dollars'] / df['amount_dollars'] * 100).round(2)
    
    plt.figure(figsize=(12, 5))
    plt.hist(df['fee_percentage'], bins=50, edgecolor='black')
    plt.title('Fee Percentage Distribution', fontsize=14, fontweight='bold')
    plt.xlabel('Fee Percentage (%)', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.show()

## 10. Success Rate Analysis

In [ ]:
# Transaction success rate (if status column exists)
if 'status' in df.columns:
    status_counts = df['status'].value_counts()
    
    print("Transaction Status Breakdown:")
    print(status_counts)
    print()
    
    # Calculate success rate
    if 'succeeded' in status_counts.index:
        success_rate = (status_counts['succeeded'] / len(df) * 100)
        print(f"Success Rate: {success_rate:.2f}%")
    
    # Pie chart
    plt.figure(figsize=(10, 6))
    plt.pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%')
    plt.title('Transaction Status Distribution', fontsize=14, fontweight='bold')
    plt.show()

## 11. Export Processed Data

In [ ]:
# Save processed data
df.to_csv('../data/processed/transactions_processed.csv', index=False)
print("Processed data saved to data/processed/transactions_processed.csv")

## Summary

This notebook provides a comprehensive analysis of your Stripe transaction data. Customize the analysis based on the specific columns in your dataset.